In [3]:
import os
import shutil
from typing import List, Dict, Any, Iterable, Set

import numpy as np
import rasterio
from rasterio.warp import transform
from rasterio.windows import Window
from rasterio.errors import RasterioIOError


def copy_rasters_if_buffer_max_exceeds_threshold(
    src_folder: str,
    lat: float,
    lon: float,
    threshold: float,
    *,
    radius_m: float = 1000.0,
    out_folder_name: str = "events_touched",
    band: int = 1,
    ignore_values: Iterable[float] = (0, 9999),
    overwrite: bool = False,
) -> List[Dict[str, Any]]:
    """
    For each .tif in src_folder:
      - locate the point
      - read a window covering `radius_m` around it
      - compute max excluding ignore_values (default: 0 and 9999)
      - if max > threshold => copy file to out folder

    Returns hit records: {path, copied_to, max_value, n_valid, row, col, crs}.
    """
    if not os.path.isdir(src_folder):
        raise FileNotFoundError(f"Folder not found: {src_folder}")

    out_dir = os.path.join(src_folder, out_folder_name)
    os.makedirs(out_dir, exist_ok=True)

    ignore_set: Set[float] = set(float(v) for v in ignore_values)

    tifs = [os.path.join(src_folder, f) for f in os.listdir(src_folder) if f.lower().endswith(".tif")]
    tifs.sort()

    hits: List[Dict[str, Any]] = []

    for fp in tifs:
        try:
            with rasterio.open(fp) as src:
                if src.crs is None:
                    continue

                # 1) point -> raster CRS (your point is WGS84)
                x, y = transform("EPSG:4326", src.crs, [lon], [lat])
                x, y = x[0], y[0]

                # 2) bounds reject
                left, bottom, right, top = src.bounds
                if not (left <= x <= right and bottom <= y <= top):
                    continue

                # 3) pixel index
                row, col = src.index(x, y)
                if row < 0 or col < 0 or row >= src.height or col >= src.width:
                    continue

                # 4) radius in pixels
                # assumes CRS units are meters (true for projected CRSs like yours)
                px_size_x, px_size_y = src.res  # (20,20)
                rad_x = int(np.ceil(radius_m / px_size_x))
                rad_y = int(np.ceil(radius_m / px_size_y))

                # 5) window (clipped to raster)
                r0 = max(row - rad_y, 0)
                r1 = min(row + rad_y + 1, src.height)
                c0 = max(col - rad_x, 0)
                c1 = min(col + rad_x + 1, src.width)

                win = Window(c0, r0, c1 - c0, r1 - r0)

                arr = src.read(band, window=win, masked=False)
                arr = arr.astype(np.float32, copy=False)

                # 6) mask ignored values (0 and 9999 by default)
                # also ignore NaNs if any
                valid_mask = np.ones(arr.shape, dtype=bool)
                for v in ignore_set:
                    valid_mask &= (arr != v)
                valid_mask &= ~np.isnan(arr)

                n_valid = int(valid_mask.sum())
                if n_valid == 0:
                    continue

                max_val = float(arr[valid_mask].max())

                if max_val > threshold:
                    dst_fp = os.path.join(out_dir, os.path.basename(fp))
                    if overwrite or not os.path.exists(dst_fp):
                        shutil.copy2(fp, dst_fp)

                    hits.append({
                        "path": fp,
                        "copied_to": dst_fp,
                        "max_value": max_val,
                        "n_valid": n_valid,
                        "row": row,
                        "col": col,
                        "crs": str(src.crs),
                        "radius_m": radius_m,
                        "window_hw": (int(win.height), int(win.width)),
                    })

        except RasterioIOError:
            continue

    return hits

In [4]:
folder = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021_filtered"
lat, lon = 50.15891380977145, 5.246305872452204

hits = copy_rasters_if_buffer_max_exceeds_threshold(
    src_folder=folder,
    lat=lat,
    lon=lon,
    threshold=1,      # example: 10 cm if your rasters are in cm
    radius_m=330,     #  km
    ignore_values=(0, 9999),
)

print("Copied:", len(hits))
for h in hits[:10]:
    print(h["max_value"], h["window_hw"], "->", h["copied_to"])


Copied: 1
10.0 (35, 35) -> D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021_filtered\events_touched\WD_MERGE_2021-07-12---2021-08-02_duration_21_days_cluster_159_A0_000619_A_000852_lat_04942_lon_00499_size_0246.tif


In [5]:
fp = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021\events_touched\WD_MERGE_2021-07-12---2021-08-02_duration_21_days_cluster_159_A0_000619_A_000852_lat_04942_lon_00499_size_0246_3857_60m_cog.tif"
import leafmap
m = leafmap.Map()
m.add_basemap("CartoDB.Positron")
m.add_marker(location=[50.15891380977145, 5.246305872452204])
m.add_raster(
    fp,
    layer_name="Flood depth (cm)",
    nodata=9999,
    opacity=0.8,
    vmin=0.1,
    vmax=10,
    cmap="Blues"   #  reversed grayscale
)

#  Pixel inspector tool
m.add("inspector")
m



Map(center=[50.7642635, 1.8357605000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

Event_ID: 002

Event_Type: Flood

Date: 2024-01

Country: France

City: Saint-Étienne-au-Mont
Site_Name: Raclot Bakery & Pastry Shop
Address: 45 Rue du Docteur Brousse, 62360 Saint-Étienne-au-Mont
Latitude: 50.680747429069655
Longitude: 1.629404533017451
Source: Press reports
Commune code : 62746

In [7]:
folder = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered"
lat, lon =  50.680747429069655, 1.629404533017451

hits = copy_rasters_if_buffer_max_exceeds_threshold(
    src_folder=folder,
    lat=lat,
    lon=lon,
    threshold=10,      # example: 10 cm rasters are in cm
    radius_m=3000,     # 1 km
    ignore_values=(0, 9999),
)

print("Copied:", len(hits))
for h in hits[:10]:
    print(h["max_value"], h["window_hw"], "->", h["copied_to"])
    m.add_marker(location=[50.680747429069655, 1.629404533017451])


Copied: 0


In [7]:
# 2 , 6

#r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\ready_plot\WD_MERGE_2024-01-01---2024-04-01_duration_91_days_cluster_332_A0_000692_A_000753_lat_05333_lon_-0217_size_0189_3857_60m_cog.tif"
import leafmap

fp = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\ready_plot\WD_MERGE_2024-01-01---2024-02-05_duration_35_days_cluster_333_A0_001861_A_002025_lat_04966_lon_00582_size_0555_3857_60m_cog.tif"

m = leafmap.Map()
m.add_basemap("CartoDB.Positron")
m.add_marker(location=[50.680747429069655, 1.629404533017451])
m.add_raster(
    fp,
    layer_name="Flood depth (cm)",
    nodata=9999,
    opacity=0.8,
    vmin=0.1,
    vmax=10,
    cmap="Blues"   #  reversed grayscale
)

#  Pixel inspector tool
m.add("inspector")
m

ImportError: localtileserver is not installed. Please install it before proceeding. https://github.com/banesullivan/localtileserver

In [21]:
import rasterio
import numpy as np

fp = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\ready_plot\WD_MERGE_2024-01-01---2024-02-05_duration_35_days_cluster_337_A0_002307_A_002648_lat_06440_lon_02916_size_0209_3857_60m_cog.tif"

with rasterio.open(fp) as src:
    a = src.read(1)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Shape:", a.shape)
    print("dtype:", a.dtype)
    print("nodata:", src.nodata)
    # quick stats excluding nodata if present
    nodata = src.nodata
    if nodata is not None:
        v = a[a != nodata]
    else:
        v = a[np.isfinite(a)]
    print("valid count:", v.size)
    if v.size:
        print("min/max:", float(v.min()), float(v.max()))


CRS: EPSG:3857
Bounds: BoundingBox(left=1483529.5493981498, bottom=7877689.757489571, right=4707449.54939815, top=10739029.757489571)
Shape: (47689, 53732)
dtype: uint16
nodata: 9999.0
valid count: 3922737
min/max: 10.0 4298.0


In [25]:
import rasterio
import numpy as np
from pyproj import Transformer

fp = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered\ready_plot\WD_MERGE_2024-01-01---2024-02-05_duration_35_days_cluster_337_A0_002307_A_002648_lat_06440_lon_02916_size_0209_3857_60m_cog.tif"

with rasterio.open(fp) as src:
    decim = src.overviews(1)[-1]  # 32
    a = src.read(1, out_shape=(src.height // decim, src.width // decim)).astype("float32")
    a[a == 9999] = np.nan

    idx = np.argwhere(np.isfinite(a))
    r, c = idx[len(idx)//2]  # take a middle valid pixel (more stable than first)
    r_full, c_full = int(r * decim), int(c * decim)

    x, y = src.transform * (c_full, r_full)  # EPSG:3857
    t = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
    lon, lat = t.transform(x, y)

    print("Suggested center (lat, lon):", lat, lon)


Suggested center (lat, lon): 64.81630046247756 25.86581674852697


In [11]:
folder = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2024_filtered"
lat, lon =  45.5859221, 4.7526672

hits = copy_rasters_if_buffer_max_exceeds_threshold(
    src_folder=folder,
    lat=lat,
    lon=lon,
    threshold=1,      # example: 10 cm rasters are in cm
    radius_m=1000,     # 1 km
    ignore_values=(0, 9999),
)

print("Copied:", len(hits))
for h in hits[:10]:
    print(h["max_value"], h["window_hw"], "->", h["copied_to"])

Copied: 0


In [5]:
import leafmap

fp = r"D:\M2_MoSEF\Acclim\DataCollection\data\JRC_flood_depth_maps\2021\events_touched\WD_MERGE_2021-07-12---2021-08-02_duration_21_days_cluster_159_A0_000619_A_000852_lat_04942_lon_00499_size_0246_3857_60m_cog.tif"

m = leafmap.Map()
m.add_basemap("CartoDB.Positron")
m.add_marker(location=[50.15891380977145, 5.246305872452204])

m.add_raster(
    fp,
    layer_name="Flood depth (cm)",
    nodata=9999,
    opacity=0.8,
    vmin=0.1,
    vmax=200,
    cmap="Blues"   #  reversed grayscale
)

#  Pixel inspector tool
m.add("inspector")
m


Map(center=[50.7642635, 1.8357605000000001], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…